# **Install packages**

In [19]:
!pip -q install transformers
!pip -q install sentencepiece
!pip -q install torch

# Download the pretrained model from Microsoft

In [3]:
from transformers import AutoTokenizer
from transformers import AutoModel

MODEL_NAME = "microsoft/graphcodebert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
print(type(model))

<class 'transformers.models.roberta.modeling_roberta.RobertaModel'>


In [22]:
total = sum(p.numel() for p in model.parameters())

print(f"Total Parameters: {total:,}")

Total Parameters: 124,645,632


# What are tokens and how are they saved?

In [23]:
code = """
int add(int a, int b)
{
    return a + b;
}
"""

tokens = tokenizer.tokenize(code)

print(tokens)

['Ċ', 'int', 'Ġadd', '(', 'int', 'Ġa', ',', 'Ġint', 'Ġb', ')', 'Ċ', '{', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb', ';', 'Ċ', '}', 'Ċ']


# Token to ID

In [24]:
ids = tokenizer.convert_tokens_to_ids(tokens)

for t, i in zip(tokens, ids):
    print(f"{t:12} -> {i}")

Ċ            -> 50118
int          -> 2544
Ġadd         -> 1606
(            -> 1640
int          -> 2544
Ġa           -> 10
,            -> 6
Ġint         -> 6979
Ġb           -> 741
)            -> 43
Ċ            -> 50118
{            -> 45152
Ċ            -> 50118
Ġ            -> 1437
Ġ            -> 1437
Ġ            -> 1437
Ġreturn      -> 671
Ġa           -> 10
Ġ+           -> 2055
Ġb           -> 741
;            -> 131
Ċ            -> 50118
}            -> 24303
Ċ            -> 50118


# Embedding

In [25]:
inputs = tokenizer(
    code,
    return_tensors="pt"
)

print(inputs)

{'input_ids': tensor([[    0, 50118,  2544,  1606,  1640,  2544,    10,     6,  6979,   741,
            43, 50118, 45152, 50118,  1437,  1437,  1437,   671,    10,  2055,
           741,   131, 50118, 24303, 50118,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]])}


In [26]:
import torch

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

torch.Size([1, 26, 768])


# Summary
Source Code
↓
Tokenizer
↓
GraphCodeBERT
↓
768-dimensional vector

Next step: Fine-tune the model

# Train the model
First, fine-tune the model

In [5]:
import torch
import torch.nn as nn

classifier = nn.Linear(
    in_features=768,
    out_features=2
)

print(classifier)

Linear(in_features=768, out_features=2, bias=True)


In [28]:
total = sum(p.numel() for p in classifier.parameters())

print(total)

1538


In [29]:
cls_vector = outputs.last_hidden_state[:, 0, :]

logits = classifier(cls_vector)

print(logits)
print(logits.shape)

tensor([[0.1767, 0.1789]], grad_fn=<AddmmBackward0>)
torch.Size([1, 2])


In [30]:
prob = torch.softmax(logits, dim=1)

print(prob)

tensor([[0.4995, 0.5005]], grad_fn=<SoftmaxBackward0>)


In [31]:
pred = torch.argmax(prob, dim=1)

print(pred)

tensor([1])


# Lesson Summary

Now we have this flow:

C/C++ Code
      │
      ▼
Tokenizer
      │
      ▼
Token IDs
      │
      ▼
GraphCodeBERT
      │
      ▼
CLS Vector (768)
      │
      ▼
Linear(768 → 2)
      │
      ▼
Logits
      │
      ▼
Softmax
      │
      ▼
Human / AI

But because we haven't trained, the returned value `tensor([0])` meaning Human or `tensor([1])` meaning AI is not correct yet. It is just a random initialization. Use `print(classifier.weight)` to see the random weights.

# Next step: Prepare training data

# Create the dataset

In [ ]:
from pathlib import Path

# Define human_files and ai_files after extraction
human_files = list(Path("data/raw/dataset/human").glob("*.c"))
ai_files = list(Path("data/raw/dataset/ai").glob("*.c"))

print(list(Path("data/raw/dataset").iterdir()))

print("Human:", len(human_files))
print("AI:", len(ai_files))

[PosixPath('dataset/human'), PosixPath('dataset/.DS_Store'), PosixPath('dataset/ai')]
Human: 62
AI: 62


In [7]:
# Now create a list of sample data

from pathlib import Path
import random

human_files = list(Path("data/raw/dataset/human").glob("*.c"))
ai_files = list(Path("data/raw/dataset/ai").glob("*.c"))

samples = []

for f in human_files:
    samples.append((f, 0))      # Human = 0

for f in ai_files:
    samples.append((f, 1))      # AI = 1

random.seed(42)
random.shuffle(samples)

print(samples[:5])

[(PosixPath('dataset/human/semihost.c'), 0), (PosixPath('dataset/ai/avl_insert_node.c'), 1), (PosixPath('dataset/ai/avl_search_node.c'), 1), (PosixPath('dataset/ai/cave_parse.c'), 1), (PosixPath('dataset/ai/fuzz_main.c'), 1)]


In [8]:
# Now split the data into training and validation sets

split = int(len(samples) * 0.8)

train_samples = samples[:split]
valid_samples = samples[split:]

print("Train:", len(train_samples))
print("Validation:", len(valid_samples))

Train: 99
Validation: 25


In [9]:
# Create a CodeDataset (standard in PyTorch projects)

from torch.utils.data import Dataset

class CodeDataset(Dataset):

    def __init__(self, samples, tokenizer, max_length=512):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        file_path, label = self.samples[idx]

        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            code = f.read()

        encoding = self.tokenizer(
            code,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": label
        }

In [10]:
train_dataset = CodeDataset(train_samples, tokenizer)

print(f"len of train_dataset = {len(train_dataset)}")

sample = train_dataset[0]

print(f"input_ids: {sample["input_ids"].shape}")
print(f"attention_mask: {sample["attention_mask"].shape}")
print(f"labels: {sample["labels"]}")

len of train_dataset = 99
input_ids: torch.Size([512])
attention_mask: torch.Size([512])
labels: 0


In [11]:
# Create a DataLoader so the GPU processes batches instead of individual files
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

valid_dataset = CodeDataset(valid_samples, tokenizer)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8,
    shuffle=False
)

In [12]:
# Verify the first batch of data

batch = next(iter(train_loader))

print(batch.keys())

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([8, 512])
torch.Size([8, 512])
torch.Size([8])


In [13]:
# Pass data through GraphCodeBERT

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

print(f"outputs.last_hidden_state.shape = {outputs.last_hidden_state.shape}")

cls_vectors = outputs.last_hidden_state[:, 0, :]

print(cls_vectors.shape)

outputs.last_hidden_state.shape = torch.Size([8, 512, 768])
torch.Size([8, 768])


In [40]:
# Pass to the classifier

logits = classifier(cls_vectors)

print(logits.shape)

torch.Size([8, 2])


In [41]:
# Apply softmax

prob = torch.softmax(logits, dim=1)

print(prob.shape)

torch.Size([8, 2])


In [42]:
# Make a prediction

pred = torch.argmax(prob, dim=1)

print(pred)

tensor([1, 1, 1, 1, 1, 1, 1, 1])


# Train the model

*   Calculate the loss
*   Compute gradients
*   Update the weights



In [14]:
# Start training

import torch
import torch.nn as nn
import torch.optim as optim

# -------------------------------------------------
# Freeze GraphCodeBERT
# -------------------------------------------------
for param in model.parameters():
    param.requires_grad = False

# -------------------------------------------------
# Loss Function
# -------------------------------------------------
criterion = nn.CrossEntropyLoss()

# -------------------------------------------------
# Optimizer
# Only train Classification Head
# -------------------------------------------------
optimizer = optim.AdamW(
    classifier.parameters(),
    lr=1e-3
)

# -------------------------------------------------
# Start training Loop
# -------------------------------------------------
for batch in train_loader:

    print("=" * 80)
    print(f"Labels: {batch['labels']}")

    # -------------------------------------------------
    # IMPORTANT:
    # Clear gradients from previous batch
    # -------------------------------------------------
    optimizer.zero_grad()

    # -------------------------------------------------
    # Forward
    # -------------------------------------------------
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"]
    )

    print(f"last_hidden_state: {outputs.last_hidden_state.shape}")

    # -------------------------------------------------
    # CLS Embedding
    # -------------------------------------------------
    cls_vectors = outputs.last_hidden_state[:, 0, :]

    print(f"CLS: {cls_vectors.shape}")

    # -------------------------------------------------
    # Classification Head
    # -------------------------------------------------
    logits = classifier(cls_vectors)

    print(f"Logits: {logits.shape}")

    # -------------------------------------------------
    # Loss
    # -------------------------------------------------
    loss = criterion(logits, batch["labels"])

    print(f"Loss: {loss.item():.6f}")

    # -------------------------------------------------
    # Check Gradient BEFORE backward
    # -------------------------------------------------
    print("\nBefore backward()")
    print("Gradient =", classifier.weight.grad)

    # -------------------------------------------------
    # Compute Gradient
    # -------------------------------------------------
    loss.backward()

    # -------------------------------------------------
    # Check Gradient AFTER backward
    # -------------------------------------------------
    print("\nAfter backward()")
    print("Gradient Shape =", classifier.weight.grad.shape)

    print(classifier.weight.grad[:2, :8])

    # -------------------------------------------------
    # Save old weight
    # -------------------------------------------------
    old_weight = classifier.weight.detach().clone()

    # -------------------------------------------------
    # Update Parameters
    # -------------------------------------------------
    optimizer.step()

    # -------------------------------------------------
    # Compare old/new weight
    # -------------------------------------------------
    changed = not torch.equal(old_weight, classifier.weight)

    print(f"\nWeight Updated: {changed}")



Labels: tensor([0, 0, 1, 1, 1, 0, 0, 1])
last_hidden_state: torch.Size([8, 512, 768])
CLS: torch.Size([8, 768])
Logits: torch.Size([8, 2])
Loss: 0.658110

Before backward()
Gradient = None

After backward()
Gradient Shape = torch.Size([2, 768])
tensor([[ 0.0209, -0.0757, -0.0132,  0.0055, -0.0094,  0.0109, -0.0315,  0.0037],
        [-0.0209,  0.0757,  0.0132, -0.0055,  0.0094, -0.0109,  0.0315, -0.0037]])

Weight Updated: True
Labels: tensor([1, 0, 0, 1, 1, 0, 1, 1])
last_hidden_state: torch.Size([8, 512, 768])
CLS: torch.Size([8, 768])
Logits: torch.Size([8, 2])
Loss: 0.603647

Before backward()
Gradient = None

After backward()
Gradient Shape = torch.Size([2, 768])
tensor([[ 0.0112, -0.0681,  0.0103,  0.0336, -0.0396, -0.0617, -0.0154,  0.0863],
        [-0.0112,  0.0681, -0.0103, -0.0336,  0.0396,  0.0617,  0.0154, -0.0863]])

Weight Updated: True
Labels: tensor([0, 0, 1, 0, 0, 0, 1, 1])
last_hidden_state: torch.Size([8, 512, 768])
CLS: torch.Size([8, 768])
Logits: torch.Size([8, 2

# Define a prediction function that we can reuse later

In [15]:
def predict_code(code):

    # Put the model in inference mode
    model.eval()
    classifier.eval()

    # tokenize
    inputs = tokenizer(
        code,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    # inference
    with torch.no_grad():

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        cls = outputs.last_hidden_state[:, 0, :]

        logits = classifier(cls)

        probs = torch.softmax(logits, dim=1)

        pred = torch.argmax(probs, dim=1).item()

    return pred, probs

# Test results

In [16]:
code = """
int add(int a, int b){
    return a+b;
}
"""

pred, probs = predict_code(code)

human_prob = probs[0][0].item()
ai_prob = probs[0][1].item()

label = "Human" if pred == 0 else "AI"

print(f"Prediction : {label}")
print(f"Human      : {human_prob:.2%}")
print(f"AI         : {ai_prob:.2%}")

Prediction : AI
Human      : 40.64%
AI         : 59.36%


# Save the trained model, then load and use it

In [17]:
# save model to file graphcodebert_human_ai.pt
import torch

torch.save({
    "graphcodebert": model.state_dict(),
    "classifier": classifier.state_dict(),
}, "graphcodebert_human_ai.pt")

In [20]:
# List saved files. If you see the file "graphcodebert_human_ai.pt", it works!
!ls -lh

In [22]:
# Reload the model and use it (graphcodebert_human_ai.pt)
import torch
import torch.nn as nn
from pathlib import Path

from transformers import AutoModel

model = AutoModel.from_pretrained("microsoft/graphcodebert-base")

classifier = nn.Linear(768, 2)

checkpoint = torch.load(
    "graphcodebert_human_ai.pt",
    map_location="cpu"
)

model.load_state_dict(checkpoint["graphcodebert"])

classifier.load_state_dict(checkpoint["classifier"])

# Make a prediction again
# code = """
# int printResult(float p) {
#   if(p <= 0.5f) {
#     return "AI";
#   } else {
#     return "Human";
#   }
# }

# void main(int *args) {
#   // Main entry
#   count << printResult(args[0]);
# }

# """

file_path = "data/raw/dataset/human/cave_mesher.c"
code = Path(file_path).read_text(encoding="utf-8")

pred, probs = predict_code(code)

human_prob = probs[0][0].item()
ai_prob = probs[0][1].item()

label = "Human" if pred == 0 else "AI"

print(f"Prediction : {label}")
print(f"Human      : {human_prob:.2%}")
print(f"AI         : {ai_prob:.2%}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Prediction : Human
Human      : 66.24%
AI         : 33.76%


# Summary


We can use GraphCodeBERT to train a model that predicts whether code is human-written or AI-generated.

Dataset
    ↓
GraphCodeBERT
    ↓
Fine-tune
    ↓
graphcodebert_human_ai.pt
    ↓
Predict

You can also load the model to use it

graphcodebert_human_ai.pt
        ↓
Export to ONNX
        ↓
Convert FP16
        ↓
ONNX Runtime
        ↓
Predict


# Next, Convert to ONNX

In [23]:
!pip install -U onnx onnxscript onnxruntime onnxconverter-common

import torch.nn as nn

class HumanAIDetector(nn.Module):

    def __init__(self, encoder, classifier):
        super().__init__()

        self.encoder = encoder
        self.classifier = classifier

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(cls)

        return logits

onnx_model = HumanAIDetector(model, classifier)

onnx_model.eval()

HumanAIDetector(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Lay

In [24]:
# Create a dummy input for export
dummy = tokenizer(
    "int main(){ return 0; }",
    return_tensors="pt",
    max_length=512,
    padding="max_length",
    truncation=True
)

In [27]:
# Export the model to ONNX

import torch

torch.onnx.export(
    onnx_model,

    (
        dummy["input_ids"],
        dummy["attention_mask"]
    ),

    "graphcodebert_human_ai.onnx",

    input_names=[
        "input_ids",
        "attention_mask"
    ],

    output_names=[
        "logits"
    ],

    dynamic_axes={
        "input_ids": {
            0: "batch",
            1: "sequence"
        },

        "attention_mask": {
            0: "batch",
            1: "sequence"
        },

        "logits": {
            0: "batch"
        }
    },

    opset_version=17,
    dynamo=False
)

/tmp/ipykernel_13987/678129815.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:212: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:78: TracerWarning: Converting a tensor to a Python boolean 

# Verify the ONNX model

In [28]:
import onnx

m = onnx.load("graphcodebert_human_ai.onnx")

onnx.checker.check_model(m)

print("ONNX OK")

ONNX OK


In [29]:
# Test the ONNX model
import onnxruntime as ort

session = ort.InferenceSession(
    "graphcodebert_human_ai.onnx"
)

inputs = tokenizer(
    code,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=512
)

result = session.run(
    None,
    {
        "input_ids": inputs["input_ids"].numpy(),
        "attention_mask": inputs["attention_mask"].numpy()
    }
)

print(result)

[array([[ 0.27958503, -0.39437297]], dtype=float32)]


In [30]:
# Convert to ONNX FP16

from onnxconverter_common import float16
import onnx

model = onnx.load(
    "graphcodebert_human_ai.onnx"
)

# convert
fp16_model = float16.convert_float_to_float16(
    model
)
#save
onnx.save(
    fp16_model,
    "graphcodebert_human_ai_fp16.onnx"
)

/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:63: UserWarning: the float32 number -inf will be truncated to -10000.0
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 7.966707293860509e-09 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -2.6527890994998415e-09 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 9.366118547404767e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -4.751529836255486e-09 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 2.0728327498886756e-08 will be 

In [31]:
# List saved files. If you see the file "graphcodebert_human_ai_fp16.onnx", it works!
!ls -lh

In [32]:
# Verify graphcodebert_human_ai_fp16.onnx
session = ort.InferenceSession(
    "graphcodebert_human_ai_fp16.onnx"
)

# Summary


*   Now we have converted and saved the model in ONNX FP16 format
*   Next, try to load and use it. Then we will be done!

Notes:

We don't need to load the GraphCodeBERT from Microsoft anymore!
After exporting, your ONNX model already contains:

GraphCodeBERT
+
Classification Head
+
All trained weights

It is now a standalone model.
The only thing you still need is the tokenizer.

Why?

Because ONNX only performs the neural network computation.

C source code
        │
        ▼
Tokenizer
        │
input_ids + attention_mask
        │
        ▼
ONNX Runtime
        │
        ▼
Logits
        │
        ▼
Softmax
        │
        ▼
Human / AI

# Load and use the ONNX FP16 model

In [33]:
# Load the tokenizer (we still need this from Microsoft)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/graphcodebert-base"
)

# Load the ONNX model
import onnxruntime as ort

session = ort.InferenceSession(
    "graphcodebert_human_ai_fp16.onnx",
    providers=["CPUExecutionProvider"]
)

# Define a prediction function using ONNX
import numpy as np

def predict_code_onnx(code):

    inputs = tokenizer(
        code,
        return_tensors="np",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    logits = session.run(
        None,
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"]
        }
    )[0]

    # Softmax
    exp = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp / exp.sum(axis=1, keepdims=True)

    pred = int(np.argmax(probs, axis=1)[0])

    label = "Human" if pred == 0 else "AI"

    return label, probs[0]

# Now make a prediction
code = """
int add(int a, int b)
{
    return a + b;
}
"""

label, probs = predict_code_onnx(code)

print(label)
print(f"Human: {probs[0]:.2%}")
print(f"AI: {probs[1]:.2%}")

# Or, predict from a C source file:
from pathlib import Path

def predict_file_onnx(file_path):

    code = Path(file_path).read_text(encoding="utf-8")

    label, probs = predict_code_onnx(code)

    print(f"File: {file_path}")
    print(f"Prediction: {label}")
    print(f"Human: {probs[0]:.2%}")
    print(f"AI: {probs[1]:.2%}")

    return label, probs

predict_file_onnx("data/raw/dataset/human/cave_mesher.c")

AI
Human: 47.71%
AI: 52.29%
File: dataset/human/cave_mesher.c
Prediction: Human
Human: 66.26%
AI: 33.76%


('Human', array([0.6626, 0.3376], dtype=float16))

# Production deployment

In production, you typically package:

human_ai_detector/
│
├── graphcodebert_human_ai_fp16.onnx
├── tokenizer/
│   ├── vocab.json
│   ├── merges.txt
│   ├── tokenizer_config.json
│   ├── special_tokens_map.json
│   └── ...
├── predictor.py
└── app.py

Instead of calling:

AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")

you would load the local tokenizer:

tokenizer = AutoTokenizer.from_pretrained("./tokenizer")

To make this work, save the tokenizer with `tokenizer.save_pretrained("./tokenizer")` for later use. This makes the model completely offline.

# We've completed the course.

Summary:

C source code
      ↓
Tokenizer
      ↓
input_ids + attention_mask
      ↓
GraphCodeBERT
      ↓
CLS embedding (768)
      ↓
Classification Head
      ↓
Logits
      ↓
Softmax
      ↓
Human / AI

Notes:

- Optimize for CPU usage.
- We also have library / other methods to reduce the steps for the real production.

You've finished the course!